# Day 3-5: 고급 기법 & 최종 최적화 (선택)

**강의 시간**: 2-3시간  
**학습 목표**:
- Data Augmentation 적용
- Model Ensemble 구축
- Test-Time Augmentation (TTA)
- Advanced Training Techniques
- 최종 Kaggle 제출 및 순위 확인

**사전 요구사항**: Day 3-4 완료  
**목표 성능**: 98.5% → 99.0%+ (Top 10% 도전)

## 🎯 0. 왜 고급 기법인가?

### 현재 상황

**Day 3-4까지의 성과:**
```
Day 3-1 (Baseline):      98.0%
Day 3-2 (Architecture):  99.1% (ResNet)
Day 3-3 (HPO):          98.5% (Custom Tuned)
Day 3-4 (Kaggle):       98.5% (제출 완료)
```

**개선 여지:**
- 현재: 98.5%
- 목표: 99.0%+ (Top 10%)
- Gap: 0.5%+ 추가 개선 필요

**4가지 전략:**
1. Data Augmentation
2. Model Ensemble
3. Test-Time Augmentation
4. Advanced Training

## 🔧 1. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
)
from tensorflow.keras.utils import to_categorical

import mlflow
import dagshub
from sklearn.model_selection import train_test_split

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU: {tf.test.is_gpu_available()}")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# MLflow 설정
import mlflow
import dagshub

repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day3-mnist-digit-recognizer')
print('✅ MLflow 설정 완료!')


### 데이터 로드

#### Google Drive 연동

In [ ]:
# from google.colab import drive

# # Google Drive 마운트
# drive.mount('/content/drive')

# print("\n✅ Google Drive 연결 완료!")
# print("📁 Drive 경로: /content/drive/MyDrive")

In [ ]:
# import os

# # 폴더 구조 생성
# # base_path = '/content/drive/MyDrive/deeplearning-bootcamp'
# base_path = '/content/drive/MyDrive/lectures/dl_bootcamp'
# day3_path = os.path.join(base_path, 'day3_mnist_digit_recognizer')
# data_path = os.path.join(day3_path, 'data/digit-recognizer')

# os.makedirs(data_path, exist_ok=True)

# print("✅ 폴더 생성 완료!")
# print(f"📁 Base: {base_path}")
# print(f"📁 Day 3: {day3_path}")
# print(f"📁 Data: {data_path}")

#### 직접 업로드

In [ ]:
import os
import zipfile
from google.colab import files

# 1. 경로 설정: /content/data/digit-recognizer 폴더 생성
# 다른 데이터와 섞이지 않게 전용 하위 폴더를 지정합니다.
base_data_path = '/content/data'
target_path = os.path.join(base_data_path, 'digit-recognizer')

os.makedirs(target_path, exist_ok=True)

# 2. 파일 업로드
print("📤 'digit-recognizer.zip' 파일을 선택해주세요...")
uploaded = files.upload()

# 3. 압축 해제 로직
zip_file_name = 'digit-recognizer.zip'

if zip_file_name in uploaded:
    print(f"\n📦 {zip_file_name}을(를) {target_path}에 압축 해제 중...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        # target_path(/content/data/digit-recognizer)에 압축 해제
        zip_ref.extractall(target_path)
    print(f"✅ 압축 해제 완료: {target_path}")

    # 세션 용량 확보를 위해 업로드된 zip 파일 삭제 (선택 사항)
    os.remove(zip_file_name)
else:
    print(f"\n⚠️ {zip_file_name} 파일이 업로드되지 않았습니다.")

# 4. 결과 확인
print(f"\n📂 {target_path} 내부 파일 목록:")
print(os.listdir(target_path))

In [ ]:
data_path = target_path

#### 데이터 로드

In [ ]:
# 데이터 & 모델 로드 (Day 3-3에서 저장한 것)
train_df = pd.read_csv(os.path.join(data_path, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'test.csv'))

# 전처리
y_train = train_df['label'].values
X_train = train_df.drop('label', axis=1).values
X_test = test_df.values

X_train = X_train / 255.0
X_test = X_test / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# Train/Val Split (Error Analysis용)
from sklearn.model_selection import train_test_split
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print(f"✅ Train: {X_train_sub.shape}")
print(f"✅ Val  : {X_val.shape}")
print(f"✅ Test : {X_test.shape}")

## 🔄 2. Data Augmentation

🔥 이 부분을 같이 작성해봅시다.

MNIST에 적합한 Augmentation 파라미터를 채워보세요.
(rotation_range, width_shift_range, height_shift_range, zoom_range 권장)

In [ ]:
# ImageDataGenerator 설정
datagen = ImageDataGenerator(
    rotation_range=   # 🔥 직접 작성이 필요합니다. (예: 10, ±10도 회전)
    width_shift_range= # 🔥 직접 작성이 필요합니다. (예: 0.1, 좌우 10% 이동)
    height_shift_range=# 🔥 직접 작성이 필요합니다. (예: 0.1, 상하 10% 이동)
    zoom_range=        # 🔥 직접 작성이 필요합니다. (예: 0.1, 10% 확대/축소)
    fill_mode='nearest'
)

print('✅ Data Augmentation 설정 완료!')
print('\n주의: horizontal_flip은 사용하지 않음 (6↔9 혼동 방지)')


In [ ]:
# Augmentation 시각화
sample_img = X_train[0:1]
sample_label = y_train[0]

fig, axes = plt.subplots(3, 3, figsize=(9, 9))

for i, ax in enumerate(axes.flat):
    if i == 0:
        ax.imshow(sample_img[0, :, :, 0], cmap='gray')
        ax.set_title('Original', fontweight='bold', fontsize=12)
    else:
        aug_img = next(datagen.flow(sample_img, batch_size=1))
        ax.imshow(aug_img[0, :, :, 0], cmap='gray')
        ax.set_title(f'Augmented {i}', fontsize=10)
    ax.axis('off')

plt.suptitle(f'Data Augmentation Examples (Label: {sample_label})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data_augmentation_examples.png', dpi=100, bbox_inches='tight')
plt.show()

## 🤝 3. Model Ensemble

### Custom Layer 정의

MLflow에서 Custom Hybrid 모델을 로드하려면 SelfAttention 클래스가 필요합니다.

In [ ]:
# SelfAttention Layer 정의 (Day 3-3에서 사용한 것)
from tensorflow.keras.layers import Layer, Conv2D
import tensorflow as tf

class SelfAttention(Layer):
    """Self-Attention Layer for CNN"""
    def __init__(self, channels, gamma_init=0.5, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.channels = channels
        self.gamma_init = gamma_init

    def build(self, input_shape):
        self.query = Conv2D(self.channels // 8, (1, 1))
        self.key = Conv2D(self.channels // 8, (1, 1))
        self.value = Conv2D(self.channels, (1, 1))

        self.gamma = self.add_weight(
            name='gamma',
            shape=(1,),
            initializer=tf.keras.initializers.Constant(self.gamma_init),
            trainable=True
        )

        super(SelfAttention, self).build(input_shape)

    def call(self, x):
        batch, height, width, channels = x.shape

        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = tf.reshape(q, [-1, height * width, self.channels // 8])
        k = tf.reshape(k, [-1, height * width, self.channels // 8])
        v = tf.reshape(v, [-1, height * width, self.channels])

        attention = tf.matmul(q, k, transpose_b=True)
        attention = tf.nn.softmax(attention, axis=-1)

        out = tf.matmul(attention, v)
        out = tf.reshape(out, [-1, height, width, self.channels])

        out = self.gamma * out + x

        return out

    def get_config(self):
        config = super().get_config()
        config.update({
            'channels': self.channels,
            'gamma_init': self.gamma_init
        })
        return config

print("✅ SelfAttention Layer 정의 완료!")

In [ ]:
# MLflow에서 Best model 로드
print("🔍 MLflow에서 Best Custom Hybrid 로드 중...")

experiment = mlflow.get_experiment_by_name('day3-mnist-digit-recognizer')

# Best Custom Hybrid 찾기
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'Best_Custom_Hybrid_Final'",
    order_by=["start_time DESC"],
    max_results=1
)

if len(runs) > 0:
    run_id = runs.iloc[0]['run_id']
    val_acc = runs.iloc[0].get('metrics.final_val_accuracy', 0)

    model_uri = f"runs:/{run_id}/model"
    best_custom = mlflow.keras.load_model(
        model_uri,
        custom_objects={'SelfAttention': SelfAttention}
    )

    print(f"✅ Best Custom Hybrid 로드 완료!")
    print(f"   Val Accuracy: {val_acc:.4f}")
else:
    raise ValueError("Best_Custom_Hybrid_Final을 찾을 수 없습니다!")

### Ensemble을 위한 추가 모델 학습

저장된 모델이 하나뿐이므로, Ensemble을 위해 ResNet과 VGG를 빠르게 학습합니다.

In [ ]:
# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, AveragePooling2D,
    Flatten, Dense, Dropout, BatchNormalization,
    GlobalAveragePooling2D, Activation, Add, Multiply,
    Reshape, Layer
)

In [ ]:
# ResNet-style 모델 빠른 학습
from tensorflow.keras.layers import Add

def residual_block(x, filters):
    fx = Conv2D(filters, (3, 3), padding='same')(x)
    fx = BatchNormalization()(fx)
    fx = Activation('relu')(fx)
    fx = Conv2D(filters, (3, 3), padding='same')(fx)
    fx = BatchNormalization()(fx)

    if x.shape[-1] != filters:
        x = Conv2D(filters, (1, 1), padding='same')(x)

    out = Add()([fx, x])
    out = Activation('relu')(out)
    return out

def build_resnet_quick():
    inputs = Input(shape=(28, 28, 1))

    x = Conv2D(32, (3, 3), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = residual_block(x, 32)
    x = MaxPooling2D((2, 2))(x)

    x = residual_block(x, 64)
    x = MaxPooling2D((2, 2))(x)

    x = residual_block(x, 128)
    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(10, activation='softmax')(x)

    return Model(inputs, outputs, name='ResNet_Quick')

print("🏗️ ResNet 모델 생성 중...")
resnet_model = build_resnet_quick()
resnet_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("🏃 ResNet 학습 중 (10 epochs)...")
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history_resnet = resnet_model.fit(
    X_train_sub, y_train_sub,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

resnet_acc = max(history_resnet.history['val_accuracy'])
print(f"\n✅ ResNet 학습 완료! Val Acc: {resnet_acc:.4f}")

In [ ]:
# VGG-style 모델 빠른 학습
def build_vgg_quick():
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Classifier
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ], name='VGG_Quick')

    return model

print("\n🏗️ VGG 모델 생성 중...")
vgg_model = build_vgg_quick()
vgg_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("🏃 VGG 학습 중 (10 epochs)...")
history_vgg = vgg_model.fit(
    X_train_sub, y_train_sub,
    batch_size=128,
    epochs=10,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

vgg_acc = max(history_vgg.history['val_accuracy'])
print(f"\n✅ VGG 학습 완료! Val Acc: {vgg_acc:.4f}")

In [ ]:
# 3개 모델 준비 완료
models = {
    'custom': best_custom,
    'resnet': resnet_model,
    'vgg': vgg_model
}

model_info = {
    'custom': {'run_name': 'Best_Custom_Hybrid_Final', 'val_acc': val_acc},
    'resnet': {'run_name': 'ResNet_Quick', 'val_acc': resnet_acc},
    'vgg': {'run_name': 'VGG_Quick', 'val_acc': vgg_acc}
}

print("\n" + "="*60)
print("  모델 준비 완료")
print("="*60)
print(f"  Custom : {val_acc:.4f}")
print(f"  ResNet : {resnet_acc:.4f}")
print(f"  VGG    : {vgg_acc:.4f}")
print("="*60)
print(f"\n🎉 총 {len(models)}개 모델로 Ensemble 구성!")

In [ ]:
# 각 모델의 Validation 성능 확인
print("="*60)
print("  개별 모델 성능 (Validation Set)")
print("="*60)

individual_accs = {}

for name, model in models.items():
    pred = model.predict(X_val, verbose=0)
    pred_labels = np.argmax(pred, axis=1)
    acc = np.mean(pred_labels == y_val)
    individual_accs[name] = acc
    print(f"{name:8s}: {acc:.4f} ({acc*100:.2f}%)")

print("="*60)

🔥 이 부분을 같이 작성해봅시다.

**Simple Average Ensemble**: 각 모델의 예측 확률을 **np.mean**으로 평균내 보세요.

In [ ]:
# Ensemble 방법 1: Simple Average
print("\n🔮 Ensemble 예측 중...")

# 각 모델의 예측
predictions = {}
for name, model in models.items():
    pred = model.predict(X_val, verbose=0)
    predictions[name] = pred

# Simple Average
ensemble_simple = # 🔥 직접 작성이 필요합니다. (np.mean(list(predictions.values()), axis=0))
ensemble_simple_labels = np.argmax(ensemble_simple, axis=1)
acc_simple = np.mean(ensemble_simple_labels == y_val)

print(f"\n✅ Simple Average Ensemble: {acc_simple:.4f} ({acc_simple*100:.2f}%)")

In [ ]:
# Ensemble 방법 2: Weighted Average
# Val Accuracy 기반 가중치
weights = {name: acc for name, acc in individual_accs.items()}
total_weight = sum(weights.values())
weights = {k: v/total_weight for k, v in weights.items()}

print("\nWeighted Average 가중치:")
for name, weight in weights.items():
    print(f"  {name:8s}: {weight:.3f}")

# Weighted Average
ensemble_weighted = sum(weights[name] * predictions[name]
                       for name in models.keys())
ensemble_weighted_labels = np.argmax(ensemble_weighted, axis=1)
acc_weighted = np.mean(ensemble_weighted_labels == y_val)

print(f"\n✅ Weighted Average Ensemble: {acc_weighted:.4f} ({acc_weighted*100:.2f}%)")

In [ ]:
# Ensemble 방법 3: Hard Voting
from scipy.stats import mode

# 각 모델의 예측 레이블
label_predictions = [np.argmax(predictions[name], axis=1)
                     for name in models.keys()]

# 다수결
ensemble_voting_labels = mode(label_predictions, axis=0)[0].flatten()
acc_voting = np.mean(ensemble_voting_labels == y_val)

print(f"✅ Hard Voting Ensemble: {acc_voting:.4f} ({acc_voting*100:.2f}%)")

In [ ]:
# Ensemble 결과 비교
ensemble_methods = {
    **{f'{name} (Individual)': acc for name, acc in individual_accs.items()},
    'Simple Average': acc_simple,
    'Weighted Average': acc_weighted,
    'Hard Voting': acc_voting
}

# 시각화
fig, ax = plt.subplots(figsize=(12, 6))

methods = list(ensemble_methods.keys())
accs = list(ensemble_methods.values())
colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

bars = ax.barh(methods, accs, color=colors, edgecolor='black')
ax.set_xlabel('Validation Accuracy', fontweight='bold', fontsize=12)
ax.set_title('Individual Models vs Ensemble Methods', fontweight='bold', fontsize=14)
ax.set_xlim(0.98, 1.0)
ax.grid(axis='x', alpha=0.3)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{acc:.4f}', va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# 최고 성능 방법
best_method = max(ensemble_methods, key=ensemble_methods.get)
best_acc = ensemble_methods[best_method]
print(f"\n🏆 Best Method: {best_method} ({best_acc:.4f})")

## 🔮 4. Test-Time Augmentation (TTA)

🔥 이 부분을 같이 작성해봅시다.

**predict_with_tta** 함수에서 원본 + 증강 예측들을 **np.mean**으로 평균내는 부분을 완성해 보세요.

In [ ]:
# TTA 함수 구현
def predict_with_tta(model, images, n_augment=5, batch_size=256):
    """Test-Time Augmentation"""

    tta_datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1
    )

    predictions = []

    # 원본 예측
    pred_original = model.predict(images, batch_size=batch_size, verbose=0)
    predictions.append(pred_original)

    # Augmented 예측
    for i in range(n_augment - 1):
        aug_images = np.array([
            tta_datagen.random_transform(img)
            for img in images
        ])

        pred_aug = model.predict(aug_images, batch_size=batch_size, verbose=0)
        predictions.append(pred_aug)

    # 평균
    final_pred = # 🔥 직접 작성이 필요합니다. (np.mean(predictions, axis=0))

    return final_pred

print("✅ TTA 함수 정의 완료!")

In [ ]:
# TTA 효과 측정 (Validation set 일부로 테스트)
print("🔮 TTA 효과 측정 중... (샘플 1000개)")

# Best 단일 모델 선택
best_single_model = models[max(individual_accs, key=individual_accs.get)]

# TTA 없이
pred_no_tta = best_single_model.predict(X_val[:1000], verbose=0)
labels_no_tta = np.argmax(pred_no_tta, axis=1)
acc_no_tta = np.mean(labels_no_tta == y_val[:1000])

# TTA 적용 (n=10)
pred_tta = predict_with_tta(best_single_model, X_val[:1000], n_augment=10)
labels_tta = np.argmax(pred_tta, axis=1)
acc_tta = np.mean(labels_tta == y_val[:1000])

print(f"\n결과:")
print(f"  Without TTA: {acc_no_tta:.4f}")
print(f"  With TTA (n=10): {acc_tta:.4f}")
print(f"  Improvement: +{(acc_tta - acc_no_tta)*100:.2f}%")

In [ ]:
# TTA 횟수별 성능 측정
print("\n🔍 최적 TTA 횟수 찾기...")

tta_results = {}

for n in [1, 3, 5, 10, 15]:
    pred = predict_with_tta(best_single_model, X_val[:1000], n_augment=n)
    labels = np.argmax(pred, axis=1)
    acc = np.mean(labels == y_val[:1000])
    tta_results[n] = acc
    print(f"  n={n:2d}: {acc:.4f}")

# 시각화
plt.figure(figsize=(10, 5))
plt.plot(list(tta_results.keys()), list(tta_results.values()),
         'bo-', linewidth=2, markersize=10)
plt.xlabel('Number of TTA Augmentations', fontweight='bold', fontsize=12)
plt.ylabel('Validation Accuracy', fontweight='bold', fontsize=12)
plt.title('TTA Performance vs Cost', fontweight='bold', fontsize=14)
plt.grid(alpha=0.3)

for n, acc in tta_results.items():
    plt.text(n, acc + 0.0002, f'{acc:.4f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('tta_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\n💡 권장: n=5~10 (성능 vs 속도 균형)")

## 🏆 5. 최종 제출

In [ ]:
# 최종 전략: Weighted Ensemble + TTA
print("🚀 최종 예측 시작...")
print("   전략: Weighted Ensemble + TTA (n=10)")
print()

final_predictions = []

for name, model in models.items():
    print(f"🔮 {name} 모델 예측 중 (TTA 적용)...")
    pred = predict_with_tta(model, X_test, n_augment=10)
    final_predictions.append(pred)

# Weighted Average
final_ensemble = sum(weights[name] * pred
                    for name, pred in zip(models.keys(), final_predictions))
final_labels = np.argmax(final_ensemble, axis=1)

print(f"\n✅ 최종 예측 완료!")
print(f"   Total predictions: {len(final_labels):,}")

In [ ]:
# Final Submission 파일 생성
final_submission = pd.DataFrame({
    'ImageId': range(1, len(final_labels) + 1),
    'Label': final_labels
})

final_submission.to_csv('final_submission_day3_5.csv', index=False)

print("\n✅ 최종 제출 파일 생성 완료!")
print("   파일명: final_submission_day3_5.csv")
print()
print("Label 분포:")
print(final_submission['Label'].value_counts().sort_index())

## 📊 6. 최종 성능 분석

In [ ]:
# Day 3 전체 여정
day3_journey = {
    'Day 3-1\nBaseline': 0.980,
    'Day 3-2\nResNet': 0.991,
    'Day 3-3\nHPO': 0.985,
    'Day 3-4\nSubmission': 0.985,
    'Day 3-5\nEnsemble+TTA': best_acc
}

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 진행 곡선
stages = list(day3_journey.keys())
accs = list(day3_journey.values())

axes[0].plot(range(len(stages)), accs, 'bo-', linewidth=2, markersize=10)
axes[0].set_xticks(range(len(stages)))
axes[0].set_xticklabels(stages, rotation=15, ha='right')
axes[0].set_ylabel('Validation Accuracy', fontweight='bold', fontsize=12)
axes[0].set_title('Day 3 Performance Journey', fontweight='bold', fontsize=14)
axes[0].set_ylim(0.975, 1.0)
axes[0].grid(alpha=0.3)

for i, (stage, acc) in enumerate(zip(stages, accs)):
    axes[0].text(i, acc + 0.001, f'{acc:.3f}', ha='center', fontsize=9)

# 개선율
baseline = 0.980
improvements = [(acc - baseline) / baseline * 100 for acc in accs]

bars = axes[1].bar(range(len(stages)), improvements,
                   color=['gray', 'blue', 'green', 'orange', 'red'],
                   edgecolor='black')
axes[1].set_xticks(range(len(stages)))
axes[1].set_xticklabels(stages, rotation=15, ha='right')
axes[1].set_ylabel('Improvement vs Baseline (%)', fontweight='bold', fontsize=12)
axes[1].set_title('Cumulative Improvement', fontweight='bold', fontsize=14)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(axis='y', alpha=0.3)

for bar, imp in zip(bars, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'+{imp:.2f}%', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Day 3 Complete Journey — 딥러닝 부트캠프',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('day3_journey.png', dpi=100, bbox_inches='tight')
plt.show()

🔥 이 부분은 수정이 필요합니다.

**run_name**을 최종 실험을 구분하기 쉬운 이름으로 채워주세요.

In [ ]:
# MLflow에 최종 실험 기록
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다.
    mlflow.log_params({
        'stage': 'day3-5',
        'techniques': 'Ensemble + TTA',
        'models': '+'.join(models.keys()),
        'ensemble_method': 'Weighted Average',
        'tta_augments': 10,
        'final_strategy': 'All techniques combined'
    })

    mlflow.log_metrics({
        'val_accuracy': best_acc,
        'baseline_improvement': (best_acc - 0.980) * 100,
        'vs_day3_4_improvement': (best_acc - 0.985) * 100
    })

    # 아티팩트
    mlflow.log_artifact('final_submission_day3_5.csv')
    mlflow.log_artifact('day3_journey.png')

    print("✅ MLflow에 최종 실험 기록 완료!")

## 🧠 7. 핵심 교훈

### Day 3 전체 요약

**기술적 성과:**
- ✅ 5가지 CNN 아키텍처 구현
- ✅ Optuna 자동 HPO
- ✅ MLflow 체계적 실험 관리
- ✅ Ensemble & TTA 적용

**학습 성과:**
- ✅ 실패한 모델 개선 경험 (Custom: 94% → 98.5%)
- ✅ 체계적 접근 방법론
- ✅ 실전 Kaggle 경험
- ✅ 고급 기법 실습

**성능 향상:**
```
Day 3-1: 98.0% (Baseline)
Day 3-5: 99.0%+ (Ensemble+TTA)

총 개선: +1.0%+
목표 달성: Top 10% 도전!
```

**각 기법의 효과:**
- Data Augmentation: +0.2~0.5%
- Ensemble: +0.3~0.7%
- TTA: +0.1~0.3%
- 총합: +0.6~1.5%

---

### Kaggle 제출

**제출 방법:**
1. https://www.kaggle.com/c/digit-recognizer 접속
2. "Submit Predictions" 클릭
3. `final_submission_day3_5.csv` 업로드
4. Public Leaderboard 점수 확인

**예상 결과:**
- Test Accuracy: 98.8~99.2%
- Public Score: 비슷한 수준
- 순위: Top 10% 목표

축하합니다! 딥러닝 부트캠프 Day 3 완료! 🎉

## ✅ Day 3-5 완료 체크리스트

- [ ] Data Augmentation 시각화
- [ ] 3개 모델 MLflow에서 로드
- [ ] 개별 모델 성능 확인
- [ ] Simple Average Ensemble
- [ ] Weighted Average Ensemble
- [ ] Hard Voting Ensemble
- [ ] Ensemble 방법 비교
- [ ] TTA 함수 구현
- [ ] TTA 효과 측정
- [ ] 최적 TTA 횟수 찾기
- [ ] 최종 예측 (Ensemble + TTA)
- [ ] final_submission_day3_5.csv 생성
- [ ] Day 3 여정 시각화
- [ ] MLflow에 최종 실험 기록
- [ ] Kaggle 재제출
- [ ] Public Leaderboard 순위 확인

## 🎉 축하합니다!

### Day 3 완료!

**당신은 이제 할 수 있습니다:**
- ✅ 다양한 CNN 아키텍처 설계 및 구현
- ✅ 체계적인 하이퍼파라미터 최적화
- ✅ MLflow로 실험 추적 및 관리
- ✅ Ensemble & TTA로 성능 극대화
- ✅ Kaggle 경진대회 참가 및 제출
- ✅ Error Analysis를 통한 모델 개선

**다음 학습 주제:**
- Day 4: 의료 이미지 분류 (ChestX-ray8)
  - Transfer Learning
  - Class Imbalance 처리
  - Multi-label Classification
  
- Day 5: 실시간 손 제스처 인식 (HaGRID)
  - 경량 모델 설계
  - 실시간 추론 최적화
  - 모바일 배포

**추가 학습 자료:**
- Kaggle Learn: https://www.kaggle.com/learn
- Fast.ai Course: https://course.fast.ai
- Deep Learning Specialization (Coursera)

수고하셨습니다! 🚀🎉